In [16]:
import pandas as pd
import random
import osmnx as ox
import geopandas as gpd
from shapely.geometry import Point

In [7]:
df = pd.read_csv("dehradun_weather_updated.csv")

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.lon, df.lat),
    crs="EPSG:4326"
)

# -------------------------
# Get OSM data
# -------------------------
place = "Dehradun, Uttarakhand, India"

rivers = ox.features_from_place(place, tags={"waterway": True})
landuse = ox.features_from_place(place, tags={"landuse": True})
buildings = ox.features_from_place(place, tags={"building": True})

# Keep valid geometries
rivers = rivers[rivers.geometry.notnull()]
landuse = landuse[landuse.geometry.notnull()]
buildings = buildings[buildings.geometry.notnull()]

# -------------------------
# Convert CRS for distance
# -------------------------
gdf = gdf.to_crs(epsg=32644)
rivers = rivers.to_crs(epsg=32644)
landuse = landuse.to_crs(epsg=32644)
buildings = buildings.to_crs(epsg=32644)

# -------------------------
# Distance to river
# -------------------------
gdf["dist_river"] = gdf.geometry.apply(
    lambda x: rivers.distance(x).min()
)

# -------------------------
# Landuse (nearest match)
# -------------------------
gdf = gpd.sjoin_nearest(
    gdf,
    landuse[["geometry", "landuse"]],
    how="left"
)

gdf["landuse"] = gdf["landuse"].fillna("unknown")

# -------------------------
# Building density
# -------------------------
gdf_buffer = gdf.copy()
gdf_buffer["geometry"] = gdf_buffer.geometry.buffer(500)

joined = gpd.sjoin(buildings, gdf_buffer, how="inner", predicate="intersects")

counts = joined.groupby("index_right").size()

gdf["building_density"] = gdf.index.map(counts).fillna(0)

# -------------------------
# Save final dataset
# -------------------------
gdf = gdf.to_crs(epsg=4326)

gdf.drop(columns="geometry").to_csv("final_dataset.csv", index=False)

print("Final dataset created successfully")

Final dataset created successfully


In [17]:
df = pd.read_csv("final_dataset.csv")

df.shape

(144, 10)

In [9]:
df.head()

,lat,lon,rainfall,temp,humidity,dist_river,element,id,landuse,building_density
0,30.25,77.95,73,307.95,10,594.047672,way,781622440,residential,0.0
1,30.25,77.97,0,307.98,10,756.136071,way,1120986164,grass,0.0
2,30.25,77.99,0,307.98,10,155.599313,way,1188616709,military,0.0
3,30.25,78.01,96,308.21,10,484.594668,way,1188616709,military,0.0
4,30.25,78.03,27,308.49,10,319.512185,way,627228950,residential,0.0


In [10]:
df.tail()

,lat,lon,rainfall,temp,humidity,dist_river,element,id,landuse,building_density
139,30.45,78.07,0,297.48,14,1677.092198,way,1374927571,residential,1.0
140,30.45,78.09,0,297.48,14,341.652849,way,1374927576,residential,4.0
141,30.45,78.11,0,297.13,15,411.439306,way,1374927576,residential,1.0
142,30.45,78.13,0,298.34,17,492.689075,way,1374927576,residential,0.0
143,30.45,78.15,0,298.34,17,21.560876,way,1291802309,construction,0.0


In [11]:
df.describe()

,lat,lon,rainfall,temp,humidity,dist_river,id,building_density
count,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,1.440000e+02,144.000000
mean,30.357361,78.040556,21.152778,305.647847,11.750000,578.893242,6.264913e+08,17.993056
std,0.063190,0.064296,31.695739,3.086629,1.484182,487.367435,4.719269e+08,80.560753
min,30.250000,77.950000,0.000000,297.130000,10.000000,4.115930,6.466214e+06,0.000000
25%,30.310000,77.990000,0.000000,304.402500,11.000000,194.703636,1.326102e+08,0.000000
50%,30.360000,78.030000,0.000000,306.855000,11.000000,459.062474,5.209223e+08,0.000000
75%,30.410000,78.090000,50.000000,307.942500,12.000000,911.669302,9.960700e+08,1.000000
max,30.450000,78.150000,100.000000,308.630000,17.000000,2263.185678,1.437089e+09,804.000000


In [18]:
# -------------------------
# 1. Drop unnecessary columns
# -------------------------
df = df.drop(columns=["element", "id"], errors="ignore")

# -------------------------
# 2. Improve rainfall (synthetic variation)
# -------------------------
def generate_rain(x):
    if x == 0:
        if random.random() < 0.6:
            return 0
        else:
            return random.randint(5, 100)
    return x

df["rainfall"] = df["rainfall"].apply(generate_rain)

# -------------------------
# 3. Create flood label
# -------------------------
def assign_flood(row):
    if row["rainfall"] > 50 and row["dist_river"] < 300:
        return 1 if random.random() > 0.2 else 0
    else:
        return 0 if random.random() > 0.2 else 1

df["flood"] = df.apply(assign_flood, axis=1)

# -------------------------
# Save final dataset
# -------------------------
df.to_csv("final_ml_dataset.csv", index=False)

print("Final ML dataset ready")

Final ML dataset ready


In [19]:
df = pd.read_csv("final_ml_dataset.csv")

df.shape

(144, 9)

In [20]:
df.head()

,lat,lon,rainfall,temp,humidity,dist_river,landuse,building_density,flood
0,30.25,77.95,73,307.95,10,594.047672,residential,0.0,0
1,30.25,77.97,31,307.98,10,756.136071,grass,0.0,0
2,30.25,77.99,0,307.98,10,155.599313,military,0.0,0
3,30.25,78.01,96,308.21,10,484.594668,military,0.0,1
4,30.25,78.03,27,308.49,10,319.512185,residential,0.0,0


In [21]:
df.describe()

,lat,lon,rainfall,temp,humidity,dist_river,building_density,flood
count,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000,144.000000
mean,30.357361,78.040556,29.868056,305.647847,11.750000,578.893242,17.993056,0.298611
std,0.063190,0.064296,32.697436,3.086629,1.484182,487.367435,80.560753,0.459246
min,30.250000,77.950000,0.000000,297.130000,10.000000,4.115930,0.000000,0.000000
25%,30.310000,77.990000,0.000000,304.402500,11.000000,194.703636,0.000000,0.000000
50%,30.360000,78.030000,19.500000,306.855000,11.000000,459.062474,0.000000,0.000000
75%,30.410000,78.090000,57.000000,307.942500,12.000000,911.669302,1.000000,1.000000
max,30.450000,78.150000,100.000000,308.630000,17.000000,2263.185678,804.000000,1.000000
